# Tech Challenge Fase 4 - Previsao de Precos de Acoes com LSTM

Este notebook apresenta a pipeline completa do projeto:
1. Coleta de dados historicos da Apple (AAPL) via Yahoo Finance
2. Analise exploratoria dos dados
3. Pre-processamento (normalizacao e criacao de sequencias)
4. Construcao e treinamento do modelo LSTM
5. Avaliacao com metricas (MAE, RMSE, MAPE)
6. Visualizacao dos resultados
7. Salvamento do modelo para uso na API

## 1. Imports e Configuracao Inicial

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import joblib
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

# Configuracao pra deixar os graficos bonitos
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 12

# Reproducibilidade - fixa a seed pra ter resultados consistentes
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

print(f'TensorFlow versao: {tf.__version__}')

## 2. Coleta de Dados

Vamos usar a biblioteca `yfinance` pra baixar o historico de precos da Apple (AAPL).
O Yahoo Finance fornece dados de:
- **Open**: preco de abertura do dia
- **High**: preco maximo do dia
- **Low**: preco minimo do dia
- **Close**: preco de fechamento (o que vamos prever)
- **Volume**: quantidade de acoes negociadas

In [ ]:
# Configuracao da coleta
SIMBOLO = 'AAPL'
DATA_INICIO = '2018-01-01'
DATA_FIM = '2024-12-31'

# Baixa os dados
df = yf.download(SIMBOLO, start=DATA_INICIO, end=DATA_FIM, progress=False)

# Achata MultiIndex se necessario
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.dropna()

print(f'Periodo: {df.index[0].date()} ate {df.index[-1].date()}')
print(f'Total de registros: {len(df)}')
print(f'\nPrimeiros registros:')
df.head()

In [ ]:
# Resumo estatistico dos dados
df.describe()

## 3. Analise Exploratoria

Antes de treinar o modelo, vamos entender os dados visualizando tendencias e padroes.

In [ ]:
# Grafico do preco de fechamento ao longo do tempo
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Preco de fechamento
axes[0].plot(df.index, df['Close'], color='#1f77b4', linewidth=1.2)
axes[0].set_title(f'Preco de Fechamento - {SIMBOLO}')
axes[0].set_ylabel('Preco (USD)')
axes[0].grid(True, alpha=0.3)

# Volume de negociacao
axes[1].bar(df.index, df['Volume'], color='#2ca02c', alpha=0.6, width=2)
axes[1].set_title(f'Volume de Negociacao - {SIMBOLO}')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Media movel de 30 e 90 dias - ajuda a ver tendencias
# A media movel "suaviza" o preco, removendo ruido do dia a dia

df['MM_30'] = df['Close'].rolling(window=30).mean()
df['MM_90'] = df['Close'].rolling(window=90).mean()

plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], label='Preco Fechamento', alpha=0.7)
plt.plot(df.index, df['MM_30'], label='Media Movel 30 dias', linewidth=2)
plt.plot(df.index, df['MM_90'], label='Media Movel 90 dias', linewidth=2)
plt.title(f'Preco de Fechamento com Medias Moveis - {SIMBOLO}')
plt.xlabel('Data')
plt.ylabel('Preco (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Retorno diario - variacao percentual de um dia pro outro
# Ajuda a entender a volatilidade da acao

df['Retorno_Diario'] = df['Close'].pct_change() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(df.index, df['Retorno_Diario'], alpha=0.7, linewidth=0.5)
axes[0].set_title('Retorno Diario (%)')
axes[0].set_ylabel('Retorno (%)')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['Retorno_Diario'].dropna(), bins=50, color='#1f77b4', edgecolor='white', alpha=0.8)
axes[1].set_title('Distribuicao dos Retornos Diarios')
axes[1].set_xlabel('Retorno (%)')
axes[1].set_ylabel('Frequencia')
axes[1].axvline(x=0, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Retorno diario medio: {df["Retorno_Diario"].mean():.4f}%')
print(f'Volatilidade (desvio padrao): {df["Retorno_Diario"].std():.4f}%')

## 4. Pre-processamento dos Dados

Agora vamos preparar os dados pro modelo LSTM:

1. **Extrair** so a coluna `Close` (preco de fechamento)
2. **Normalizar** entre 0 e 1 com MinMaxScaler (redes neurais funcionam melhor assim)
3. **Dividir** em treino (80%) e teste (20%) - sem embaralhar!
4. **Criar sequencias** com janela deslizante (60 dias -> prever o proximo)

In [ ]:
# Pega so o preco de fechamento
precos = df['Close'].values.reshape(-1, 1)

print(f'Total de dados: {len(precos)}')
print(f'Preco minimo: ${precos.min():.2f}')
print(f'Preco maximo: ${precos.max():.2f}')
print(f'Preco medio: ${precos.mean():.2f}')

In [ ]:
# Normalizacao com MinMaxScaler
# Transforma os valores pro intervalo [0, 1]
# Isso e necessario porque redes neurais convergem melhor com valores pequenos.
# Se nao normalizar, gradientes ficam muito grandes e o treino nao funciona direito.

scaler = MinMaxScaler(feature_range=(0, 1))
dados_normalizados = scaler.fit_transform(precos)

print(f'Antes: min={precos.min():.2f}, max={precos.max():.2f}')
print(f'Depois: min={dados_normalizados.min():.4f}, max={dados_normalizados.max():.4f}')

In [ ]:
# Split treino/teste: 80% treino, 20% teste
# Importante: NAO embaralhamos! Em series temporais a ordem cronologica importa.
# Treinamos com dados antigos e testamos com dados recentes (simula o mundo real).

proporcao_treino = 0.8
tamanho_treino = int(len(dados_normalizados) * proporcao_treino)

dados_treino = dados_normalizados[:tamanho_treino]
dados_teste = dados_normalizados[tamanho_treino:]

print(f'Dados de treino: {len(dados_treino)} ({proporcao_treino*100:.0f}%)')
print(f'Dados de teste: {len(dados_teste)} ({(1-proporcao_treino)*100:.0f}%)')

# Visualiza a divisao
plt.figure(figsize=(14, 4))
plt.plot(range(len(dados_treino)), dados_treino, label='Treino', color='blue')
plt.plot(range(len(dados_treino), len(dados_normalizados)), dados_teste, label='Teste', color='red')
plt.axvline(x=len(dados_treino), color='black', linestyle='--', label='Divisao')
plt.title('Divisao Treino/Teste (dados normalizados)')
plt.ylabel('Preco Normalizado')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Criacao de sequencias com janela deslizante
# O modelo recebe os precos dos ultimos 60 dias e tenta prever o proximo.
#
# Exemplo simplificado (janela=3):
#   dados = [10, 20, 30, 40, 50]
#   X[0] = [10, 20, 30] -> y[0] = 40
#   X[1] = [20, 30, 40] -> y[1] = 50

TAMANHO_JANELA = 60  # usar 60 dias anteriores pra prever o proximo

def criar_sequencias(dados, tamanho_janela):
    """Cria sequencias de entrada (X) e saida (y) usando janela deslizante."""
    X, y = [], []
    for i in range(tamanho_janela, len(dados)):
        X.append(dados[i - tamanho_janela:i, 0])
        y.append(dados[i, 0])
    
    X = np.array(X)
    y = np.array(y)
    
    # LSTM espera entrada 3D: (amostras, timesteps, features)
    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, y

X_treino, y_treino = criar_sequencias(dados_treino, TAMANHO_JANELA)
X_teste, y_teste = criar_sequencias(dados_teste, TAMANHO_JANELA)

print(f'X_treino shape: {X_treino.shape} -> (amostras, timesteps, features)')
print(f'y_treino shape: {y_treino.shape}')
print(f'X_teste shape: {X_teste.shape}')
print(f'y_teste shape: {y_teste.shape}')

## 5. Construcao do Modelo LSTM

### Arquitetura

O LSTM (Long Short-Term Memory) e um tipo de rede neural recorrente que consegue
aprender padroes de longo prazo em dados sequenciais. Diferente de RNNs comuns,
o LSTM tem um mecanismo de portas que controla o fluxo de informacao:

- **Forget Gate**: decide o que esquecer da memoria anterior
- **Input Gate**: decide o que guardar de novo
- **Output Gate**: decide o que mandar como saida

Nosso modelo:
```
Input (60 dias, 1 feature)
  -> LSTM (50 neuronios) + Dropout (20%)
  -> LSTM (50 neuronios) + Dropout (20%)
  -> Dense (25 neuronios, ReLU)
  -> Dense (1 neuronio) -> Preco previsto
```

In [ ]:
# Montando o modelo
modelo = Sequential([
    # Define o formato da entrada
    Input(shape=(TAMANHO_JANELA, 1)),
    
    # Primeira camada LSTM com 50 neuronios
    # return_sequences=True porque a proxima camada tambem e LSTM
    # e precisa receber a sequencia completa
    LSTM(50, return_sequences=True),
    Dropout(0.2),  # desliga 20% dos neuronios aleatoriamente (regularizacao)
    
    # Segunda camada LSTM com 50 neuronios
    # return_sequences=False porque a proxima e Dense e espera um vetor unico
    LSTM(50, return_sequences=False),
    Dropout(0.2),
    
    # Camadas densas pra gerar a previsao
    Dense(25, activation='relu'),
    Dense(1)  # saida: 1 valor (o preco previsto do proximo dia)
])

# Compilacao:
# - adam: otimizador que ajusta a taxa de aprendizado automaticamente
# - mean_squared_error: funcao de perda padrao pra problemas de regressao
modelo.compile(optimizer='adam', loss='mean_squared_error')

modelo.summary()

## 6. Treinamento do Modelo

In [ ]:
# Configuracao do treinamento
EPOCHS = 50       # numero maximo de vezes que o modelo ve todos os dados
BATCH_SIZE = 32   # quantas amostras processar antes de atualizar os pesos

# EarlyStopping: para o treinamento automaticamente se o modelo parar de melhorar.
# patience=5 significa que se o loss da validacao nao melhorar em 5 epochs seguidas,
# o treinamento para e volta pros melhores pesos encontrados.
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Treina o modelo
historico = modelo.fit(
    X_treino, y_treino,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,  # 10% do treino vira validacao (pra monitorar overfitting)
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
# Evolucao do loss durante o treinamento
# O ideal e que tanto o loss de treino quanto o de validacao diminuam juntos.
# Se o loss de treino diminui mas o de validacao aumenta, e sinal de overfitting.

plt.figure(figsize=(10, 4))
plt.plot(historico.history['loss'], label='Loss Treino', linewidth=2)
plt.plot(historico.history['val_loss'], label='Loss Validacao', linewidth=2)
plt.title('Evolucao do Loss durante o Treinamento')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Treinamento parou na epoch {len(historico.history["loss"])}')
print(f'Melhor loss de validacao: {min(historico.history["val_loss"]):.6f}')

## 7. Avaliacao do Modelo

Vamos avaliar o modelo no conjunto de teste usando 3 metricas:

- **MAE (Mean Absolute Error)**: erro medio absoluto em dolares. "Em media, erramos $X"
- **RMSE (Root Mean Square Error)**: similar ao MAE, mas penaliza erros grandes. Util pra ver se o modelo tem erros discrepantes.
- **MAPE (Mean Absolute Percentage Error)**: erro medio em %. "Erramos X% em media"

In [ ]:
# Faz previsoes no conjunto de teste
previsoes_normalizadas = modelo.predict(X_teste)

# Volta pra escala original (dolares)
previsoes = scaler.inverse_transform(previsoes_normalizadas)
reais = scaler.inverse_transform(y_teste.reshape(-1, 1))

# Calcula as metricas
mae = mean_absolute_error(reais, previsoes)
rmse = np.sqrt(mean_squared_error(reais, previsoes))

# MAPE - erro percentual medio
mascara = reais.flatten() != 0
mape = np.mean(np.abs((reais.flatten()[mascara] - previsoes.flatten()[mascara]) / reais.flatten()[mascara])) * 100

print('=== Metricas de Avaliacao ===')
print(f'MAE  (Erro Absoluto Medio):     ${mae:.2f}')
print(f'RMSE (Raiz do Erro Quadratico): ${rmse:.2f}')
print(f'MAPE (Erro Percentual Medio):   {mape:.2f}%')
print()
print(f'Interpretacao: em media, a previsao erra ${mae:.2f} ({mape:.2f}%) do valor real.')

In [ ]:
# Grafico: previsao vs preco real
plt.figure(figsize=(14, 5))
plt.plot(reais, label='Preco Real', color='blue', linewidth=1.5)
plt.plot(previsoes, label='Previsao LSTM', color='red', linewidth=1.5, alpha=0.8)
plt.title(f'Previsao LSTM vs Preco Real - {SIMBOLO} (Conjunto de Teste)')
plt.xlabel('Dias')
plt.ylabel('Preco (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Grafico de dispersao - mostra a correlacao entre valores reais e previstos
# Quanto mais perto da linha diagonal, melhor o modelo

plt.figure(figsize=(6, 6))
plt.scatter(reais, previsoes, alpha=0.5, s=10)

# Linha diagonal (previsao perfeita)
min_val = min(reais.min(), previsoes.min())
max_val = max(reais.max(), previsoes.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Previsao Perfeita')

plt.title('Valores Reais vs Previstos')
plt.xlabel('Preco Real (USD)')
plt.ylabel('Preco Previsto (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Distribuicao dos erros de previsao
erros = reais.flatten() - previsoes.flatten()

plt.figure(figsize=(10, 4))
plt.hist(erros, bins=40, color='#1f77b4', edgecolor='white', alpha=0.8)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Erro zero')
plt.axvline(x=erros.mean(), color='green', linestyle='--', linewidth=2, label=f'Media: ${erros.mean():.2f}')
plt.title('Distribuicao dos Erros de Previsao')
plt.xlabel('Erro (Real - Previsto) em USD')
plt.ylabel('Frequencia')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Erro medio: ${erros.mean():.2f}')
print(f'Desvio padrao do erro: ${erros.std():.2f}')

## 8. Salvamento do Modelo

Salvamos dois artefatos:
- **Modelo** (`.keras`): os pesos e a arquitetura da rede neural
- **Scaler** (`.joblib`): o normalizador, necessario pra converter os dados de entrada e saida da API

Ambos sao necessarios na hora de fazer inferencia (previsao) na API.

In [ ]:
# Salva o modelo e o scaler
DIRETORIO_MODELOS = Path('../models')
DIRETORIO_MODELOS.mkdir(exist_ok=True)

caminho_modelo = DIRETORIO_MODELOS / 'lstm_model.keras'
caminho_scaler = DIRETORIO_MODELOS / 'scaler.joblib'

modelo.save(caminho_modelo)
print(f'Modelo salvo em: {caminho_modelo}')

joblib.dump(scaler, caminho_scaler)
print(f'Scaler salvo em: {caminho_scaler}')

## 9. Teste Rapido da Previsao

Vamos simular o que a API faz: receber os ultimos 60 precos e prever o proximo.

In [ ]:
# Pega os ultimos 60 precos reais
ultimos_precos = precos[-TAMANHO_JANELA:]

# Normaliza
ultimos_normalizados = scaler.transform(ultimos_precos)

# Formata pro LSTM: (1 amostra, 60 timesteps, 1 feature)
entrada = ultimos_normalizados.reshape(1, TAMANHO_JANELA, 1)

# Faz a previsao
previsao = modelo.predict(entrada, verbose=0)
preco_previsto = scaler.inverse_transform(previsao)[0, 0]

ultimo_preco_real = float(precos[-1])
variacao = ((preco_previsto - ultimo_preco_real) / ultimo_preco_real) * 100

print(f'Ultimo preco real: ${ultimo_preco_real:.2f}')
print(f'Previsao proximo dia: ${preco_previsto:.2f}')
print(f'Variacao esperada: {variacao:+.2f}%')

## 10. Conclusao

### Resultados
O modelo LSTM foi capaz de capturar a tendencia geral dos precos da AAPL, com metricas
de erro dentro de uma faixa aceitavel para series temporais financeiras.

### Limitacoes
- O mercado financeiro e influenciado por fatores externos (noticias, politica, etc.) que
  o modelo nao considera
- O modelo usa apenas o preco de fechamento como feature. Adicionar volume, indicadores
  tecnicos ou dados de sentimento poderia melhorar os resultados
- Previsoes de longo prazo (varios dias a frente) tendem a ser menos precisas

### Proximos Passos
- Testar diferentes tamanhos de janela (30, 90, 120 dias)
- Adicionar mais features (volume, medias moveis, RSI)
- Experimentar arquiteturas mais profundas ou bidirecionais
- Implementar re-treinamento periodico do modelo